In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: GEO02 - Business Travel
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_unit
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. High-Risk Countries: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. AMEX Data: hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
   5. CWT Data: hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
   
   LOGIC SUMMARY:
   1. MASTER AU LIST: Uses fy25_list_of_unit as the anchor. Every single BusinessID 
      will print in the final output.
   2. HIGH RISK: Filters td_country_risk_rating_abac for 'High' rating.
   3. TRAVEL DATA: Unions AMEX and CWT, casting Numberoftrips to INT.
   4. FILTERING & MAPPING: Joins to high risk countries, then maps to AU_ID using 
      vw_cost_center_mapping_bootstrap. Sums trips per AU.
   5. FINAL OUTPUT: LEFT JOINS the aggregated trips back to the Master AU List. 
      Outputs '0' if no trips occurred.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Combined_Travel_Data AS (
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        TRY_CAST(TRIM(Numberoftrips) AS INT) AS Trip_Count,
        'AMEX' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
    
    UNION ALL
    
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        TRY_CAST(TRIM(Numberoftrips) AS INT) AS Trip_Count,
        'CWT' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
),

High_Risk_Trips AS (
    SELECT 
        t.Cleaned_CC,
        t.DestinationCountry,
        t.Trip_Count
    FROM Combined_Travel_Data t
    INNER JOIN High_Risk_Countries h
        ON t.DestinationCountry = h.CountryName
    WHERE t.Cleaned_CC IS NOT NULL
),

Trips_By_AU AS (
    SELECT 
        m.AU_ID,
        SUM(h.Trip_Count) AS Total_Trips
    FROM vw_cost_center_mapping_bootstrap m
    INNER JOIN High_Risk_Trips h
        ON CAST(m.Cost_Center_ID AS INT) = h.Cleaned_CC
    GROUP BY m.AU_ID
)

SELECT 
    u.BusinessID,                          
    u.AU_Name,                             
    'GEO02' AS QuestionID,               
    
    COALESCE(CAST(t.Total_Trips AS STRING), '0') AS Response
    
FROM Master_AUs u
LEFT JOIN Trips_By_AU t 
    ON u.BusinessID = t.AU_ID
ORDER BY u.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: GEO02 - Business Travel Drill-Down
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_unit
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. High-Risk Countries: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. AMEX Data: hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
   5. CWT Data: hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(business_ID AS STRING)) AS BusinessID,
        TRIM(business_operational_unit_name) AS AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_unit
    WHERE business_ID IS NOT NULL
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Combined_Travel_Data AS (
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        TRY_CAST(TRIM(Numberoftrips) AS INT) AS Trip_Count,
        'AMEX' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
    
    UNION ALL
    
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        UPPER(TRIM(DestinationCountry)) AS DestinationCountry,
        TRY_CAST(TRIM(Numberoftrips) AS INT) AS Trip_Count,
        'CWT' AS Source_System
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
),

Mapped_Trips AS (
    SELECT 
        m.AU_ID,
        m.Cost_Center_ID,
        t.Source_System,
        t.DestinationCountry,
        t.Trip_Count
    FROM vw_cost_center_mapping_bootstrap m
    INNER JOIN Combined_Travel_Data t
        ON CAST(m.Cost_Center_ID AS INT) = t.Cleaned_CC
    INNER JOIN High_Risk_Countries h
        ON t.DestinationCountry = h.CountryName
)

SELECT 
    u.BusinessID,
    u.AU_Name,
    mt.Cost_Center_ID AS Mapped_Cost_Center,
    mt.Source_System,
    mt.DestinationCountry AS High_Risk_Destination,
    mt.Trip_Count
    
FROM Master_AUs u
LEFT JOIN Mapped_Trips mt 
    ON u.BusinessID = mt.AU_ID
ORDER BY u.BusinessID;